# 4. Comparing models, carefully

This dataset cannot answer "which model is better". Models were not assigned to
sessions at random; they were chosen, and chosen for reasons that correlate with
exactly the things we would want to measure.

What follows is a description of how each model was used, with the confounder
stated rather than hidden.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DATA = Path.cwd().parent / "data"
sessions = pd.read_parquet(DATA / "sessions.parquet")
events = pd.read_parquet(DATA / "events.parquet")
print(f"{len(sessions):,} sessions, {len(events):,} events")

1,213 sessions, 181,406 events


In [2]:
from agent_telemetry.analysis import models

profile = models.model_profile(sessions)
profile

,provider,primary_model,sessions,median_events,median_duration,median_tool_calls,median_output_tokens,mean_cache_hit_rate,mean_tool_error_rate
0,deepseek,deepseek-v4-flash,912,31.0,2.20,10.0,12965.5,0.9572,0.0220
1,anthropic,claude-sonnet-5,102,113.0,11.70,29.0,30373.0,0.8831,0.0282
2,unknown,,54,8.5,0.00,0.0,0.0,0.0000,0.0000
3,deepseek,deepseek-v4-pro,38,64.0,3.25,20.5,8741.0,0.8769,0.0352
4,anthropic,claude-opus-4-8,28,99.5,11.65,18.0,22338.5,0.8343,0.0363
5,unknown,other,15,34.0,7.10,0.0,164.0,0.3075,0.0365
6,synthetic,<synthetic>,13,11.0,2.60,0.0,0.0,0.0000,0.0000
7,anthropic,claude-opus-5,11,197.0,18.10,45.0,38070.0,0.9365,0.0498
8,anthropic,claude-haiku-4-5-20251001,9,4.0,0.40,0.0,1161.0,0.4791,0.0000
9,local,qwen/qwen3.5-4b,7,32.0,10.00,4.0,1038.0,0.4122,0.1488


Note the providers. The logs mix hosted Anthropic models, a third party API and
locally served models. Those are not comparable on cost, latency or capability,
and grouping them into one table would be the first mistake available here.

In [3]:
profile.groupby("provider", as_index=False).agg(
    models=("primary_model", "nunique"),
    sessions=("sessions", "sum"),
)

,provider,models,sessions
0,anthropic,5,156
1,deepseek,2,950
2,local,3,18
3,other,1,7
4,synthetic,1,13
5,unknown,2,69


## The confounder, stated plainly

If a model is picked for hard problems, it will show longer sessions and more
tool calls regardless of how good it is. The correlation below is therefore
uninterpretable as quality.

In [4]:
comparison = profile[profile["sessions"] >= 10][
    ["primary_model", "sessions", "median_events", "median_tool_calls", "median_output_tokens"]
]
comparison

,primary_model,sessions,median_events,median_tool_calls,median_output_tokens
0,deepseek-v4-flash,912,31.0,10.0,12965.5
1,claude-sonnet-5,102,113.0,29.0,30373.0
2,,54,8.5,0.0,0.0
3,deepseek-v4-pro,38,64.0,20.5,8741.0
4,claude-opus-4-8,28,99.5,18.0,22338.5
5,other,15,34.0,0.0,164.0
6,<synthetic>,13,11.0,0.0,0.0
7,claude-opus-5,11,197.0,45.0,38070.0


## The closest thing to a controlled comparison

Sessions where the model was switched partway through. Both models saw related
work, which removes some of the selection effect, and introduces another one:
switching usually happens because the first model was not doing well.

In [5]:
models.paired_sessions(sessions).head(10)

,models,sessions,median_events,median_tool_calls,median_duration
0,<synthetic>+claude-sonnet-5,14,2020.0,350.0,180.2


There are too few of these to conclude anything. Reporting the count is the
honest version of the analysis.

## Reasoning effort

Effort is set by the caller, so this is a record of habits rather than of model
behaviour.

In [6]:
effort = models.effort_distribution(events)
effort[effort["events"] >= 50].head(20)

,model,effort,events,share
0,claude-fable-5,low,735,0.7394
1,claude-fable-5,high,259,0.2606
2,claude-opus-4-8,low,1267,0.5391
3,claude-opus-4-8,xhigh,744,0.3166
4,claude-opus-4-8,max,267,0.1136
5,claude-opus-4-8,high,72,0.0306
6,claude-opus-5,low,1195,0.6940
7,claude-opus-5,high,501,0.2909
9,claude-sonnet-5,xhigh,7011,0.4047
10,claude-sonnet-5,low,5895,0.3403
